# Extract merged-cell evidence

Use the reusable extraction controls from source; no pipeline loading or crop implementation is duplicated here.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks" or PROJECT_ROOT.parent.name == "notebooks":
    while PROJECT_ROOT.name != "notebooks":
        PROJECT_ROOT = PROJECT_ROOT.parent
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
from src.io import PipelinePaths

SAMPLE_ID = "44b6_0113de3b"
FRAME = 0
paths = PipelinePaths.discover(PROJECT_ROOT)
sample_path = paths.sample_zarr(SAMPLE_ID)


In [ ]:
import napari
import pandas as pd
from src.io import load_npy_time_series, load_processed_dataset_inputs, open_sample
from diagnostics.cell_volume_extraction import add_cell_volume_extractor

inputs = load_processed_dataset_inputs(SAMPLE_ID, paths=paths)
raw = open_sample(sample_path)
cells = pd.concat([table.assign(frame=i) for i, table in enumerate(inputs.time_frames)], ignore_index=True)
preprocessed, _ = load_npy_time_series(inputs.root / "preprocessing")
mask, _ = load_npy_time_series(inputs.root / "masking")
labels, _ = load_npy_time_series(inputs.root / "segmentation")
viewer = napari.Viewer(ndisplay=3)
viewer.add_image(preprocessed, name="preprocessed")
viewer.add_labels(labels, name="instances", visible=False)
extractor = add_cell_volume_extractor(
    viewer=viewer,
    cells=cells,
    image_volume=raw,
    sample_id=SAMPLE_ID,
    preprocessed_volume=preprocessed,
    binary_mask_volume=mask,
    instance_labels_volume=labels,
)
viewer
